In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install flash-attention

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes torch

In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

In [ ]:
DATASET_PATH = {
    "reasoning": "/content/drive/MyDrive/Resilio/prompts/reasoning_prompts.json",
    "logical": "/content/drive/MyDrive/Resilio/prompts/logical_prompts.json",
    "classification": "/content/drive/MyDrive/Resilio/prompts/classification_prompts.json",
    "qna": "/content/drive/MyDrive/Resilio/prompts/qna_prompts.json"
}

MODELS = {
    "llama":   "meta-llama/Meta-Llama-3-8B-Instruct",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "phi":     "microsoft/Phi-3-mini-4k-instruct",
    "qwen": "Qwen/Qwen2.5-1.5B-Instruct",
    "gptoss": "openai/gpt-oss-20b"
}

QUANTIZATION_LEVELS = ["fp16", "8bit", "4bit"]

In [ ]:
OUTPUT_ROOT = "/content/drive/MyDrive/Resilio/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:
def load_model(model_name, quant_level):
    print(f"\nLoading {model_name} with {quant_level}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    if quant_level == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        )

    elif quant_level == "8bit":
        # Wrap 8-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config
        )

    elif quant_level == "4bit":
        # Wrap 4-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config
        )
        print("successfully loaded!!")

    else:
        raise ValueError("Invalid quantization level. Choose from 'fp16', '8bit', or '4bit'.")

    return tokenizer, model

In [ ]:
temp = 0.5
top_p = 0.9
max_token = 384
rep_pen = 1.2

SYSTEM_PROMPT = """You are a knowledgeable assistant. Answer the question directly and concisely.
You are a precise and concise AI assistant.
Rules:
- The response MUST be under 256 tokens.
- Aim for 150–220 tokens when possible.
- Do not exceed 256 tokens under any circumstances.
- No filler, no repetition, no unnecessary explanations.
- Be direct, structured, and information-dense.
- Use short paragraphs or bullet points when helpful.
- Do not restate the question.
- Do not include disclaimers unless explicitly required.
- End immediately after completing the answer.
If the answer would exceed 256 tokens, summarize aggressively to stay within limit.
"""

def generate_response(tokenizer, model, prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_token,
            temperature=temp,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=rep_pen,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = outputs[0][input_len:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


# changes below

In [ ]:
MODEL = MODELS["phi"]   # llama, mistral, phi, qwen, gptoss
QUANT = QUANTIZATION_LEVELS[2]  #0-fp16, 1-8bit, 2-4bit

tokenizer, model = load_model(MODEL, QUANT)

output_folder = os.path.join(OUTPUT_ROOT, f"{MODEL}_{QUANT}")
os.makedirs(output_folder, exist_ok=True)


Loading microsoft/Phi-3-mini-4k-instruct with 4bit...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

successfully loaded!!


In [ ]:
task_category = "reasoning"     # reasoning, logical, classification, qna


DATASET_PATH = DATASET_PATH[task_category]
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

prompts = dataset["prompts"]

print(f"Loaded {len(prompts)} prompts")


Loaded 160 prompts


In [ ]:
output_folder_for_task = os.path.join(output_folder, f"{task_category}")
os.makedirs(output_folder_for_task, exist_ok=True)

checkpoint_path = os.path.join(output_folder_for_task, "responses.jsonl")
final_json_path = os.path.join(output_folder_for_task, "responses.json")

for item in tqdm(prompts):
    prompt_text = item["prompt"]

    try:
        response = generate_response(tokenizer, model, prompt_text)
    except Exception as e:
        response = f"ERROR: {str(e)}"
    print(f"-> {response[:50]}")

    # Save immediately to prevent data loss
    with open(checkpoint_path, "a") as f:
        result = {**item, "model": MODEL, "quantization": QUANT, "response": response}
        f.write(json.dumps(result) + "\n")

print(f"Process complete. Responses saved to {checkpoint_path}")


final_results = []
try:
    with open(checkpoint_path, "r") as f:
        for line in f:
            final_results.append(json.loads(line))

    with open(final_json_path, "w") as f:
        json.dump(final_results, f, indent=4)

    print(f"\n✅ All responses processed and saved to a clean JSON list at: {final_json_path}")

    # Optional: Remove the .jsonl checkpoint if you only want the final JSON
    # os.remove(checkpoint_path)

except Exception as e:
    print(f"Conversion failed, but your data is safe in {checkpoint_path}. Error: {e}")


  1%|          | 1/160 [00:24<1:04:57, 24.51s/it]

-> To address this issue effectively while adhering s


  1%|▏         | 2/160 [00:48<1:03:29, 24.11s/it]

-> To address recurrent blackouts amidst increasing e


  2%|▏         | 3/160 [01:12<1:03:28, 24.26s/it]

...

-> To address recurrent electricity disruativas cause


  4%|▍         | 6/160 [02:25<1:01:59, 24.15s/it]

-> To address recurrent electricity supply issues due


  4%|▍         | 6/160 [02:39<1:08:20, 26.63s/it]


KeyboardInterrupt: 

In [ ]:
# Memory Cleanup
del model
del tokenizer
torch.cuda.empty_cache()

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2
